## **Artigo: Automatic Tracking of Hyoid Bone Displacement and Rotation Relative to Cervical Vertebrae in Videofluoroscopic Swallow Studies Using Deep Learning** 

### **Configurações**

In [2]:
# Importando Bibliotecas
import sys
sys.path.append('../../')

import os
import pandas as pd
import re
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

# Importando Modelos
from src.models.artigo_full_hrnet.full_hrnet_pretrained import FullHRNet_ImageNet
from src.models.artigo_full_hrnet.full_hrnet import FullHRNet

# Importar sua classe existente
from src.vfss_dataset import VFSSImageDataset

from src.split_data import split_data_k_fold
from src.utils import (plot_image_with_mask,
                       get_script_relative_path,
                       get_project_root_directory,
                       get_corners_from_angle,
                       clahe,
                       modify_input,
                       load_points,
                       custom_collate_fn,
                       set_seed,
                       avalia_status_GPU,
                       resolve_dataframe_path)

# Importando classes/funções - Podar imports que não são chamados aqui.
from src.training.config import TrainingConfig
from src.training.loss import (LossCalculator,
                               FocalMSELoss,
                               FocalMSEMaskedLoss)
from src.training.holdout import holdout
from src.training.cross_validation import cross_validate
from src.evalutation.inference import evaluate_model_on_test

set_seed(42)

In [ ]:
# Parametros Básicos Iniciais
epochs = 200

model_class = FullHRNet
criterion_heatmap = nn.MSELoss()
criterion_roi = nn.BCEWithLogitsLoss()

model_name = f"{model_class.__name__}\\{criterion_heatmap.__class__.__name__}_{str(epochs)}ep"
print("="*30, f"\n{model_name}\n", "="*30)

model_kwargs = {
    "num_keypoints": 2,
}

# Configuração
config = TrainingConfig(
    learning_rate=0.001,
    batch_size=8,
    epochs=epochs,
    criterion_roi = criterion_roi,
    criterion_heatmap= criterion_heatmap,
    patience=5,
    checkpoint_dir=f"data\\model_weights\\{model_name}",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Caminhos Importantes
videos_dir = 'data\\videos\\' 
labels_dir = 'data\\rotulos\\anotacoes-tecgraf\\'
frames_dir = 'data\\frames'
augmentation_dir = 'data/frames_augmentation'
path_dataframe = "data/metadados/video_frame_metadata.csv"

root = get_project_root_directory()
path_dataframe = os.path.join(root, path_dataframe)
video_frame_df = pd.read_csv(path_dataframe)
video_frame_df = video_frame_df.apply(resolve_dataframe_path, axis = 1)

# Adicionando os pontos direto no dataframe, para não precisar ficar abrindo arquivos
video_frame_df["keypoints"] = video_frame_df["target_dir"].apply(load_points)

# Atributos da função VFSSImageDataset
sigma = 20
output_dim = (448, 448)
offline_augmentation = True

transform_augmentation_offline = A.Compose([
    A.GaussianBlur(blur_limit=(3, 3), sigma_limit=(0.5, 0.5), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0, contrast_limit=(0.75 - 1, 1.5 - 1), p=0.5),
    # Augmentations Geométricas (Position Transformations) 
    A.ShiftScaleRotate(
        shift_limit=0.10,      # Tradução de +/- 10% 
        scale_limit=(-0.5, 0.2), # Escala de 50% a 120% (0.5 a 1.2) 
        rotate_limit=25,       # Rotação de +/- 25 graus 
        interpolation=cv2.INTER_LINEAR,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=0.5
    ),
    A.Affine(shear=(-10, 10), p=0.5),       # Cisalhamento (Shearing) de +/- 10 graus 
    A.Resize(448, 448), 
], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))

transform_puro = A.Compose([A.Resize(output_dim[0], output_dim[1]),
                       ToTensorV2()],
                      keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))

# Aplicando a divisão dos Folds
print("\nSplit Dados")
list_df_folds, df_test = split_data_k_fold(video_frame_df, test_size=0.2, n_folds=5)

# Avaliando Status do Uso da GPU
avalia_status_GPU()

### **Treinamento/Validação**

In [ ]:
results = holdout(
    model_class=model_class,
    df_train=pd.concat([list_df_folds[j] for j in range(1,5)], ignore_index=True),
    df_val=list_df_folds[0],
    config=config,
    output_dim=output_dim,
    modify_input_fn=modify_input,
    dataset_class=VFSSImageDataset,
    transform_augmentation = transform_augmentation_offline, # transforma para augmentation offline no treino
    transform_train = transform_puro, # Usar transform de online augmentation (leve se com offline aug) ou sem augmentation
    transform_validation = transform_puro, # Usar transform sem augmentation para validação
    sigma = sigma,
    collate_fn=custom_collate_fn,
    offline_augmentation = offline_augmentation,
    augmentation_dir = augmentation_dir,
    model_kwargs=model_kwargs
)

### **Teste**